# 05 Embedding Generation with gte-Qwen2-1.5B

`04_hierarchical_chunking.ipynb` で生成した `data/chunks_hier.parquet` を読み込み、
`gte-Qwen2-1.5B-instruct` で 1536 次元の埋め込みベクトルを生成する。

## 方針

- **対象**: 9 ticker × (10-K × 3 年 + 10-Q × 4 Q) の Item 1A + Item 7 の全 chunk
- **モデル**: `Alibaba-NLP/gte-Qwen2-1.5B-instruct` (last-token pooling, 1536 dim)
- **デバイス**: M3 Mac MPS + bfloat16 (約 3 GB VRAM)
- **保存形式**:
    - `data/embeddings.npy`: float32 行列 (N, 1536), L2 正規化済み
    - `data/chunks_meta.parquet`: chunk_id をキーにしたメタデータ (text 除く)

## 本 notebook のスコープ (Step 1-2)

1. モデル + データロード
2. **小バッチ試走 (100 chunk, batch_size=8)** — メモリ・速度を実測
3. 全件エンコード — 試走で問題なければ実行 (次のセッションで追加)

## 既存 03-2 notebook との違い

| 項目        | 03-2 (旧)                                          | 05 (本 notebook)                                     |
| ----------- | -------------------------------------------------- | ---------------------------------------------------- |
| 入力        | `chunks.parquet` (固定 510 tok, FinBERT tokenizer) | `chunks_hier.parquet` (≤512 tok, Qwen2 tokenizer)    |
| 対象 ticker | AAPL/MSFT (2 社)                                   | 9 社 (AAPL/MSFT/GOOGL/NVDA/TSLA/AVGO/AMAT/AMZN/META) |
| max_length  | 2048                                               | 512 (実トークンと一致)                               |
| 保存        | parquet に vector を埋め込み                       | npy + meta parquet に分離                            |
| メトリクス  | なし                                               | 所要時間・RAM 実測                                   |


In [1]:
# Cell 1: imports + 環境設定
# HuggingFace のキャッシュはデフォルト (~/.cache/huggingface/) を使用する。
# 03-2 で既に gte-Qwen2-1.5B-instruct (6.6 GB) がそこにあるため、再ダウンロードを避ける。
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn.functional as F
from IPython.display import HTML, Markdown, display
from tqdm.auto import tqdm

# データ保存先 (notebook と同階層の data/)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def get_device() -> torch.device:
    """M3 Mac なら MPS、なければ CPU フォールバック."""
    if torch.backends.mps.is_available():
        print("Using Apple Silicon GPU (MPS)")
        return torch.device("mps")
    print("MPS not available. Falling back to CPU.")
    return torch.device("cpu")


def mem_used_gb() -> float:
    """現在のプロセスの RSS メモリ使用量 (GB)."""
    return psutil.Process().memory_info().rss / 1024**3


device = get_device()
print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"Initial RAM: {mem_used_gb():.2f} GB")

Using Apple Silicon GPU (MPS)
DATA_DIR: /Users/yukihata/Desktop/quants/notebook/FILING_NLP/data
Initial RAM: 0.28 GB


In [2]:
# Cell 2: モデル + トークナイザロード
# gte-Qwen2 は trust_remote_code=True 必須 (modeling_qwen.py 同梱).
# bfloat16 + MPS で約 3 GB VRAM。03-2 で既に HF キャッシュにあれば 10 秒程度で完了。
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="mps",
    low_cpu_mem_usage=True,
)
model.eval()
print(f"Model loaded. RAM: {mem_used_gb():.2f} GB")

Loading model: Alibaba-NLP/gte-Qwen2-1.5B-instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded. RAM: 0.11 GB


In [3]:
# Cell 3: chunks_hier.parquet をロード (04_hierarchical_chunking.ipynb の出力)
df_chunks = pd.read_parquet(DATA_DIR / "chunks_hier.parquet")

# chunk_id を採番 (行順で 6 桁 zero-pad)
df_chunks = df_chunks.reset_index(drop=True)
df_chunks["chunk_id"] = df_chunks.index.astype(str).str.zfill(6)

display(df_chunks.head())

print(f"total chunks: {len(df_chunks):,}")
print(
    f"tickers ({df_chunks['ticker'].nunique()}): "
    f"{df_chunks['ticker'].unique().tolist()}"
)
print(f"forms: {df_chunks['form'].unique().tolist()}")
print(f"item_keys: {df_chunks['item_key'].unique().tolist()}")

print("\n=== token_count 分布 ===")
print(df_chunks["token_count"].describe().round(1).to_string())

,filing_id,ticker,form,filing_date,item_key,subsection_idx,subsection_title,chunk_idx,text,token_count,chunk_id
0,0000320193-25-000079,AAPL,10-K,2025-10-31,item_1a,0,Item 1A. Risk Factors,0,The following summarizes factors that could ha...,120,000000
1,0000320193-25-000079,AAPL,10-K,2025-10-31,item_1a,1,Macroeconomic and Industry Risks,0,The Company’s operations and performance depen...,449,000001
2,0000320193-25-000079,AAPL,10-K,2025-10-31,item_1a,1,Macroeconomic and Industry Risks,1,"The Company has a large, global business with ...",509,000002
3,0000320193-25-000079,AAPL,10-K,2025-10-31,item_1a,1,Macroeconomic and Industry Risks,2,"For example, the U.S. Department of Commerce h...",142,000003
4,0000320193-25-000079,AAPL,10-K,2025-10-31,item_1a,1,Macroeconomic and Industry Risks,3,"Many of the Company’s operations, retail store...",490,000004


total chunks: 4,490
tickers (9): ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'TSLA', 'AVGO', 'AMAT', 'AMZN', 'META']
forms: ['10-K', '10-Q']
item_keys: ['item_1a', 'item_7']

=== token_count 分布 ===
count    4490.0
mean      275.2
std       178.5
min         2.0
25%        97.0
50%       287.0
75%       457.0
max       840.0


## エンコード方針

### Last-token Pooling

gte-Qwen2 は **decoder-only モデル** なので、文の意味は **最終トークンの hidden state** に圧縮される (causal LM 由来)。Mean pooling は decoder-only では効きが悪いので使わない。

実装上の注意:

- HuggingFace tokenizer は **right-padding** がデフォルト → 各サンプルの最終 non-pad トークン位置は `attention_mask.sum(dim=1) - 1`
- left-padding が使われている場合は単純に `[:, -1]` で取れる (両対応にしておく)

### Instruction Prefix の扱い

gte-Qwen2-instruct は **クエリ側のみ** `"Instruct: {task}\nQuery: {text}"` プレフィックスを付けると検索精度が上がる。
今回はドキュメント (10-K 本文) の埋め込みのみなので **prefix は付けない**。後段で検索クエリを作る場合のみ別途付ける。

### 既知の互換性問題

`transformers 4.45+` で `DynamicCache.get_usable_length()` が廃止されたが、gte-Qwen2 同梱の `modeling_qwen.py` はまだ古い API を呼ぶ。embedding 用途では KV キャッシュ不要なので **`use_cache=False`** を渡してそのコードパスを完全に回避する。


In [6]:
# Cell 4: エンコードヘルパー
def last_token_pool(
    last_hidden_states: torch.Tensor, attention_mask: torch.Tensor
) -> torch.Tensor:
    """各サンプルで最終 non-pad トークンの hidden state を取り出す.

    Parameters
    ----------
    last_hidden_states : torch.Tensor
        shape (batch, seq, hidden) のモデル出力.
    attention_mask : torch.Tensor
        shape (batch, seq) の attention mask.

    Returns
    -------
    torch.Tensor
        shape (batch, hidden) の pooled vector.
    """
    left_padding = bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item())
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device), seq_lens
    ]


@torch.inference_mode()
def encode_texts(
    texts: list[str],
    *,
    max_length: int = 512,
    batch_size: int = 8,
    show_progress: bool = True,
) -> np.ndarray:
    """テキストのリストを 1536 次元の正規化済みベクトルにエンコード.

    Parameters
    ----------
    texts : list[str]
        対象テキスト (instruction prefix は呼び出し側で制御).
    max_length : int
        トークン上限. chunks_hier.parquet は 512 tok 以下なので 512 でちょうど.
    batch_size : int
        MPS では 1.5B bfloat16 で 8 程度が安全 (16 GB Mac の場合).
    show_progress : bool
        tqdm 表示の有無.

    Returns
    -------
    np.ndarray
        shape (len(texts), 1536), float32, L2 normalized.
    """
    all_vecs: list[np.ndarray] = []
    iterator = range(0, len(texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="encode")
    for start in iterator:
        batch = texts[start : start + batch_size]
        enc = tokenizer(
            batch,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)
        # use_cache=False で modeling_qwen.py の DynamicCache 互換性問題を回避
        out = model(**enc, use_cache=False, return_dict=True)
        vecs = last_token_pool(out.last_hidden_state, enc["attention_mask"])
        vecs = F.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.to(torch.float32).cpu().numpy())
    return np.vstack(all_vecs)


print("helpers ready")

helpers ready


In [ ]:
# Cell 5: 小バッチ試走 (100 chunk, batch_size=8) で M3 の性能を実測
# 全件エンコードの所要時間と RAM 使用量を見積もる。
SAMPLE_N = 500
BATCH_SIZE = 32
MAX_LENGTH = 512

sample = (
    df_chunks.sample(n=SAMPLE_N, random_state=42)
    .sort_values("token_count")
    .reset_index(drop=True)
)

print(f"=== サンプル {SAMPLE_N} chunk の token 分布 ===")
print(sample["token_count"].describe().round(1).to_string())
print(
    f"（参考）全chunkのtoken分布：\n  mean={df_chunks['token_count'].mean():.1f},\n  max={df_chunks['token_count'].max():.1f}"
)

ram_before = mem_used_gb()
print(f"\nRAM before encoding: {ram_before:.2f} GB")
print(f"settings: batch_size={BATCH_SIZE}, max_length={MAX_LENGTH}, device={device}")

t0 = time.perf_counter()
sample_vecs = encode_texts(
    sample["text"].tolist(),
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
)
elapsed = time.perf_counter() - t0
ram_after = mem_used_gb()

ms_per_chunk = elapsed / SAMPLE_N * 1000
norms = np.linalg.norm(sample_vecs, axis=1)

print("\n=== 結果 ===")
print(f"  shape:           {sample_vecs.shape}")
print(f"  dtype:           {sample_vecs.dtype}")
print(
    f"  norms:           min={norms.min():.4f} max={norms.max():.4f} "
    f"mean={norms.mean():.4f} (期待値: ≈ 1.0)"
)
print(f"  elapsed:         {elapsed:.2f} s")
print(f"  speed:           {ms_per_chunk:.1f} ms/chunk")
print(f"  RAM delta:       {ram_after - ram_before:+.2f} GB (now: {ram_after:.2f} GB)")

# 全件エンコードの見積もり
total_chunks = len(df_chunks)
est_total_s = elapsed / SAMPLE_N * total_chunks
est_size_mb = total_chunks * 1536 * 4 / 1024**2
print(f"\n=== 全 {total_chunks:,} chunk の見積もり ===")
print(f"  所要時間:    {est_total_s:.0f} 秒 ({est_total_s / 60:.1f} 分)")
print(f"  出力サイズ:  {est_size_mb:.1f} MB (float32, embeddings.npy)")

=== サンプル 500 chunk の token 分布 ===
count    500.0
mean     283.6
std      179.2
min        2.0
25%      106.8
50%      292.5
75%      465.2
max      694.0

RAM before encoding: 0.07 GB
settings: batch_size=32, max_length=512, device=mps


encode:   0%|          | 0/16 [00:00<?, ?it/s]

## 試走結果の判定基準

| 指標        | 期待値 / 安全圏       | 問題があれば                                               |
| ----------- | --------------------- | ---------------------------------------------------------- |
| `shape`     | `(100, 1536)`         | モデルロードかバッチ処理を疑う                             |
| `dtype`     | `float32`             | OK (内部 bfloat16 → 出力時に変換)                          |
| `norms`     | 全要素 ≈ 1.0 (±0.001) | L2 正規化が効いている                                      |
| `RAM delta` | < 4 GB                | `batch_size=8` 維持                                        |
| `RAM delta` | 4-6 GB                | `batch_size=4` に下げて再試走                              |
| `RAM delta` | > 6 GB                | 16 GB Mac では不安、`batch_size=2`                         |
| `ms/chunk`  | 100-300 ms            | 順当 (MPS が効いている)                                    |
| `ms/chunk`  | > 500 ms              | MPS が CPU fallback している可能性 (`device` 表示を再確認) |

## 次のステップ

試走が問題なければ、次のセッションで以下を追加する:

1. 全 chunk エンコード (`encode_texts(df_chunks['text'].tolist())`)
2. `embeddings.npy` + `chunks_meta.parquet` への保存
3. 健全性チェック (norm 分布、ticker 別ベクトル分散など)

その後 `06_chamfer_similarity.ipynb` を作成し、対称・非対称 Chamfer 両方を計算する。


In [ ]:
# Cell 6: 全件エンコード (リジューム対応) + 保存
#
# 設計:
#   - meta は先に text 込みで保存 (途中クラッシュしても meta だけは残る)
#   - embeddings.npy は NaN マーカーで未処理 chunk を管理
#   - CHECKPOINT_EVERY_N_BATCHES バッチごとに npy へ保存 (進捗の永続化)
#   - 再実行時: 既存 embeddings.npy の shape/dtype を検証し、NaN 行から resume
#   - FORCE_REENCODE=True にすると既存ファイルを無視して再生成

BATCH_SIZE = 32
MAX_LENGTH = 512
CHECKPOINT_EVERY_N_BATCHES = 10  # 約 3 分ごと (batch=32, 約 19 s/batch)
FORCE_REENCODE = False  # True にすると既存 embeddings.npy を捨てて再生成

EMBEDDINGS_PATH = DATA_DIR / "embeddings.npy"
META_PATH = DATA_DIR / "chunks_meta.parquet"

N = len(df_chunks)
HIDDEN = 1536


def _try_resume() -> tuple[np.ndarray, int]:
    """既存 embeddings.npy から resume を試みる. 失敗時は新規 NaN 行列を返す.

    Returns
    -------
    tuple[np.ndarray, int]
        (embeddings 配列, 次に処理すべき行 index)
    """
    if FORCE_REENCODE or not EMBEDDINGS_PATH.exists():
        return np.full((N, HIDDEN), np.nan, dtype=np.float32), 0
    try:
        existing = np.load(EMBEDDINGS_PATH)
    except Exception as e:
        print(f"  既存 embeddings.npy 読み込み失敗 ({e}). 新規作成.")
        return np.full((N, HIDDEN), np.nan, dtype=np.float32), 0
    if existing.shape != (N, HIDDEN) or existing.dtype != np.float32:
        print(
            f"  既存ファイルの shape/dtype 不一致 "
            f"({existing.shape}/{existing.dtype} vs ({N}, {HIDDEN})/float32). 新規作成."
        )
        return np.full((N, HIDDEN), np.nan, dtype=np.float32), 0
    has_nan = np.isnan(existing).any(axis=1)
    if not has_nan.any():
        return existing, N
    first_unprocessed = int(np.where(has_nan)[0][0])
    return existing, first_unprocessed


# meta を text 込みで先に保存 (途中クラッシュでも meta だけは残る)
df_chunks.to_parquet(META_PATH, index=False)
meta_size_mb = META_PATH.stat().st_size / 1024**2
print(f"meta saved: {META_PATH} ({meta_size_mb:.1f} MB, text 列含む)")

# embeddings の resume 判定
embeddings, resume_idx = _try_resume()

if resume_idx == N:
    print(f"既に全 {N:,} chunk エンコード済み。スキップ。")
    print(
        f"  embeddings: {EMBEDDINGS_PATH} ({EMBEDDINGS_PATH.stat().st_size / 1024**2:.1f} MB)"
    )
else:
    if resume_idx == 0:
        print(f"新規エンコード開始: {N:,} chunks")
    else:
        print(
            f"resume: chunk {resume_idx:,} / {N:,} から再開 "
            f"(前回までに {resume_idx:,} 件処理済み)"
        )

    texts = df_chunks["text"].tolist()
    ram_before = mem_used_gb()
    t_start = time.perf_counter()
    batches_since_ckpt = 0
    batch_starts = list(range(resume_idx, N, BATCH_SIZE))

    with torch.inference_mode():
        for start in tqdm(batch_starts, desc="encode"):
            end = min(start + BATCH_SIZE, N)
            batch_texts = texts[start:end]
            enc = tokenizer(
                batch_texts,
                max_length=MAX_LENGTH,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(device)
            out = model(**enc, use_cache=False, return_dict=True)
            vecs = last_token_pool(out.last_hidden_state, enc["attention_mask"])
            vecs = F.normalize(vecs, p=2, dim=1)
            embeddings[start:end] = vecs.to(torch.float32).cpu().numpy()

            batches_since_ckpt += 1
            if batches_since_ckpt >= CHECKPOINT_EVERY_N_BATCHES:
                np.save(EMBEDDINGS_PATH, embeddings)
                batches_since_ckpt = 0

    # 最終保存
    np.save(EMBEDDINGS_PATH, embeddings)
    elapsed = time.perf_counter() - t_start
    ram_after = mem_used_gb()
    processed_n = N - resume_idx

    print("\n=== 完了 ===")
    print(f"  処理 chunk:     {processed_n:,}")
    print(f"  elapsed:        {elapsed:.1f} s ({elapsed / 60:.1f} 分)")
    print(f"  speed:          {elapsed / processed_n * 1000:.1f} ms/chunk")
    print(
        f"  RAM delta:      {ram_after - ram_before:+.2f} GB (now: {ram_after:.2f} GB)"
    )
    print(
        f"  embeddings:     {EMBEDDINGS_PATH} ({EMBEDDINGS_PATH.stat().st_size / 1024**2:.1f} MB)"
    )

meta saved: data/chunks_meta.parquet (2.9 MB, text 列含む)
新規エンコード開始: 4,490 chunks


encode:   0%|          | 0/141 [00:00<?, ?it/s]

In [ ]:
# Cell 7: エンコード結果の健全性チェック
# Cell 6 完了後に実行する。embeddings.npy と chunks_meta.parquet を読み直して検証する。
embeddings_loaded = np.load(EMBEDDINGS_PATH)
meta_loaded = pd.read_parquet(META_PATH)

print("=== ファイル ===")
print(
    f"  embeddings.npy:      shape={embeddings_loaded.shape}, dtype={embeddings_loaded.dtype}"
)
print(
    f"  chunks_meta.parquet: rows={len(meta_loaded)}, "
    f"size={META_PATH.stat().st_size / 1024**2:.2f} MB"
)
print(f"  meta columns: {list(meta_loaded.columns)}")
assert len(embeddings_loaded) == len(meta_loaded), "embeddings と meta の行数が不一致"

print("\n=== ベクトル健全性 ===")
nan_mask = np.isnan(embeddings_loaded).any(axis=1)
n_nan = int(nan_mask.sum())
print(f"  未処理 (NaN) 行数: {n_nan} (期待値: 0)")
if n_nan == 0:
    norms = np.linalg.norm(embeddings_loaded, axis=1)
    print(
        f"  norms: min={norms.min():.4f}, max={norms.max():.4f}, "
        f"mean={norms.mean():.4f} (期待値: ≈ 1.0)"
    )
    near_zero = int((norms < 0.99).sum())
    print(f"  norm < 0.99 の chunk 数: {near_zero} (期待値: 0)")

print("\n=== 件数集計 (ticker × form × item_key) ===")
print(meta_loaded.groupby(["ticker", "form", "item_key"]).size().to_string())

if n_nan == 0:
    print("\n=== ベクトル意味的妥当性チェック ===")
    # AAPL Item 1A 内 vs AAPL Item 1A × Item 7 のセクション識別性
    mask_a1 = (
        (meta_loaded["ticker"] == "AAPL") & (meta_loaded["item_key"] == "item_1a")
    ).values
    mask_a7 = (
        (meta_loaded["ticker"] == "AAPL") & (meta_loaded["item_key"] == "item_7")
    ).values
    v_a1 = embeddings_loaded[mask_a1]
    v_a7 = embeddings_loaded[mask_a7]
    intra_a1 = float((v_a1 @ v_a1.T).mean())
    intra_a7 = float((v_a7 @ v_a7.T).mean())
    inter_a1_a7 = float((v_a1 @ v_a7.T).mean())
    print(f"  AAPL Item 1A 内 cosine 平均:       {intra_a1:.4f}")
    print(f"  AAPL Item 7  内 cosine 平均:       {intra_a7:.4f}")
    print(f"  AAPL Item 1A vs Item 7 cosine 平均: {inter_a1_a7:.4f}")
    print(
        f"  intra - inter = {min(intra_a1, intra_a7) - inter_a1_a7:+.4f} "
        f"(>0 期待: セクション区別できている)"
    )

    # ticker 識別性 (AAPL Item 1A vs MSFT Item 1A の cosine 平均)
    mask_m1 = (
        (meta_loaded["ticker"] == "MSFT") & (meta_loaded["item_key"] == "item_1a")
    ).values
    v_m1 = embeddings_loaded[mask_m1]
    inter_a1_m1 = float((v_a1 @ v_m1.T).mean())
    print(f"  AAPL vs MSFT (Item 1A) cosine 平均: {inter_a1_m1:.4f}")
    print(
        f"  intra(AAPL Item 1A) - inter(AAPL vs MSFT Item 1A) = "
        f"{intra_a1 - inter_a1_m1:+.4f} (>0 期待: ticker 区別できている)"
    )